# Ejercicio 1 · Rectificación métrica — versión paso a paso

Notebook de trabajo incremental: cada punto se añade tras explicarlo en el chat.

**Regla del ejercicio:** núcleo en NumPy, sin OpenCV (OpenCV solo al final, para comparar).

| Punto | Estado |
|---|---|
| 0. Imports y configuración | ✅ |
| 1. Carga de la imagen | ✅ |
| 2. Puntos origen (4 esquinas) | ⏳ |
| 3. Puntos destino | ⏳ |
| 4. Homografía (DLT + SVD + Hartley) | ⏳ |
| 5. Warp propio + verificación de paralelismo | ⏳ |
| 6. Comparación con OpenCV | ⏳ |

## 0. Imports y configuración

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

np.set_printoptions(suppress=True, precision=4)

# ---------------- CONFIGURACIÓN ----------------
IMG_PATH  = "foto_suelo.jpg"   # tu foto, en la misma carpeta que este notebook
MAX_WIDTH = 1000               # ancho máximo (px) con el que trabajaremos
# -----------------------------------------------

## 1. Carga de la imagen

Leemos la foto, corregimos la orientación EXIF del móvil, la reducimos a `MAX_WIDTH` y la convertimos en un array NumPy `img` de forma `(alto, ancho, 3)`.

In [ ]:
if not os.path.exists(IMG_PATH):
    raise FileNotFoundError(f"No encuentro '{IMG_PATH}'. Copia tu foto junto a este notebook o cambia IMG_PATH.")

im = Image.open(IMG_PATH)
print("Tamaño original:", im.size, "| modo:", im.mode)

im = ImageOps.exif_transpose(im)      # aplica la rotación que guarda el móvil en los metadatos
im = im.convert("RGB")                # garantiza 3 canales (descarta alfa / escala de grises)

SCALE = MAX_WIDTH / im.width if im.width > MAX_WIDTH else 1.0
if SCALE < 1.0:
    im = im.resize((round(im.width * SCALE), round(im.height * SCALE)), Image.LANCZOS)

img = np.asarray(im)                  # array (alto, ancho, 3), uint8
H_IMG, W_IMG = img.shape[:2]

print(f"Factor de reducción SCALE = {SCALE:.4f}")
print(f"img: shape={img.shape}, dtype={img.dtype}, min={img.min()}, max={img.max()}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(img)
ax.set_xticks(range(0, W_IMG, 100)); ax.set_yticks(range(0, H_IMG, 100))
ax.grid(True, alpha=0.4)
ax.set_xlabel("x (columnas, px)"); ax.set_ylabel("y (filas, px)")
ax.set_title(f"Imagen cargada: {W_IMG} x {H_IMG} px")
plt.show()